# Silver: expeditions_members
Cleans and transforms `himalaya.bronze.expeditions_peaks` into `himalaya.silver.expeditions_peaks`

##Table Overview

In [0]:
import pyspark.sql.functions as F
from datetime import date

In [0]:
df = spark.table("himalaya.bronze.expeditions_peaks")

In [0]:
df.printSchema()

In [0]:
display(df.limit(3))

# Simplify Data

  > ## Drop irrelevant columns

In [0]:
drop_cols = [
    "pkname2", "heightf", "unlisted", "trekking", "trekyear",
    "restrict", "phost", "pmonth", "pday", "pexpid", "psmtnote", "psummiters"
]

df = df.drop(*drop_cols)

In [0]:
print(df.columns)

In [0]:
display(df.limit(5))

  > ## Distinct Data

In [0]:
for col in df.columns:
    df.select(col).distinct().show()

  > ## Casting Columns

In [0]:
df = df.withColumn("pyear", F.col("pyear").cast("int"))

  > ## Rename Columns

In [0]:
df = df.withColumnsRenamed({
    "pkname": "peak_name",
    "heightm": "height_m",
    "himal": "himalayan_range",
    "pstatus": "status",
    "pyear": "first_ascent_year",
    "pseason": "first_ascent_season",
    "pcountry": "first_ascent_country"
})

  > ## Reorder Data

In [0]:
df = df.select(
    "peakid", "peak_name", "location", "height_m",
    "himalayan_range", "region", "open", "status",
    "first_ascent_year", "first_ascent_season", "first_ascent_country",
    "ingested_at"
)

In [0]:
display(df.limit(5))

> ## Silver Transfer

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("himalaya.silver.expeditions_peaks")
print("✅ Written to himalaya.silver.expeditions_peaks")

In [0]:
display(spark.table("himalaya.silver.expeditions_peaks").limit(5))

# Transformations Applied

| Column | Transformation |
|---|---|
| `pkname` | Renamed to `peak_name` |
| `heightm` | Renamed to `height_m` |
| `himal` | Renamed to `himalayan_range` |
| `pstatus` | Renamed to `status` |
| `pyear` | Cast from Double to Integer, renamed to `first_ascent_year` |
| `pseason` | Renamed to `first_ascent_season` |
| `pcountry` | Renamed to `first_ascent_country` |
| `psummiters` | Dropped — free text names, not useful for analysis |
| 11 columns | Dropped — not relevant to analysis (alternate names, height in feet, trekking flags, restrictions, internal references) |
| Column order | Reordered logically by category |